## Test our Finetuned model

In [ ]:
!pip install -q --upgrade bitsandbytes==0.49 trl==0.25.1 transformers peft bitsandbytes accelerate trl datasets huggingface_hub

In [ ]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")

login(token=HF_TOKEN)

In [ ]:
# Upload from local machine
from google.colab import files
uploaded = files.upload()  # Upload comparison_results.pkl

## Load the finetuned model from Huggingface

In [ ]:
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed
from peft import PeftModel

BASE_MODEL = "meta-llama/Llama-3.2-3B"
# Quant Config
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

# Load model and tokenizer

tokenizer = AutoTokenizer.from_pretrained("davemathews/housing_pricer")

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=quant_config, device_map="auto"
)
base_model.resize_token_embeddings(len(tokenizer))

fine_tuned_model = PeftModel.from_pretrained(base_model, "davemathews/housing_pricer")

set_seed(42)

# Get our testing data

In [ ]:
from data import load_raw_data, filter_data, create_splits, PRICE_MAX, PRICE_MIN, create_description, parse_price

df_raw = load_raw_data()
df_clean = filter_data(df_raw)
train_df, val_df, test_df = create_splits(df_clean, train_size=500, val_size=50, test_size=50)
test_df['description'] = test_df.apply(create_description, axis=1)


# Run the model for inference using a sample of our test dat

In [ ]:
from tqdm import tqdm
sample_test = test_df.head(10)
sample_y_test = sample_test['price'].values


fine_tuned_llama_preds = []
for _, row in tqdm(sample_test.iterrows(), total=len(sample_test), desc="Fine-tuned LLAMA"):
    input = tokenizer.encode(
        f"What is the estimated price of this property?\n\n{row['description']}",
        return_tensors="pt"
    ).to(fine_tuned_model.device)

    with torch.no_grad():
        outputs = fine_tuned_model.generate(input, max_new_tokens=7)
    response = tokenizer.decode(outputs[0])
    pred = parse_price(response.split('assistant')[-1].strip())
    fine_tuned_llama_preds.append(pred)

# Compare the results from our week 6 exercise

In [ ]:
# Get results from week 6
import pickle

# Load (Unpickle) the array back into memory
with open('comparison_results.pkl', 'rb') as f:
    results = pickle.load(f)

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np


# Compute metrics from the predictions
def compute_metrics(y_true, y_pred) -> dict:
    """Compute MAE, RMSE, MAPE.

    Args:
        y_true: Actual prices
        y_pred: Predicted prices

    Returns:
        Dictionary with mae, rmse, mape keys
    """
    y_true, y_pred = np.array(y_true), np.array(y_pred)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    mask = y_true != 0
    if mask.sum() > 0:
        mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    else:
        mape = 0.0

    return {
        "mae": mae,
        "rmse": rmse,
        "mape": mape
    }

metrics = compute_metrics(sample_y_test, fine_tuned_llama_preds)
results.append({"name": "Fine-tuned LLAMA", "type": "LLM", **metrics})
print(f"Fine-tuned LLAMA - MAE: ${metrics['mae']:,.0f}, RMSE: ${metrics['rmse']:,.0f}, MAPE: {metrics['mape']:.1f}%")

In [ ]:
print("\n" + "="*60)
print("MODEL COMPARISON SUMMARY")
print("="*60)
print(f"{'Model':<30} {'Type':<15} {'MAE':>12} {'MAPE':>8}")
print("-"*60)
for r in sorted(results, key=lambda x: x['mae']):
    print(f"{r['name']:<30} {r['type']:<15} ${r['mae']:>10,.0f} {r['mape']:>7.1f}%")
print("="*60)